# Error handling (exceptions)
* until now we always assumed our code runs without problems - real programs need to expect and handle errors gracefully
* in python, errors happen as "exceptions" - a special object that is "raised" when something goes wrong and interrupts the normal program flow
* if an exception is not handled, the program crashes and prints a "traceback" (the red error text you probably already saw a few times by now)

In [1]:
# lets provoke a few classic exceptions on purpose, to learn to recognize their names
try:
    print(1 / 0)
except ZeroDivisionError as error:
    print("caught:", type(error).__name__, "-", error)

try:
    my_list = [1, 2, 3]
    print(my_list[10])
except IndexError as error:
    print("caught:", type(error).__name__, "-", error)

try:
    my_dict = {"a": 1}
    print(my_dict["b"])
except KeyError as error:
    print("caught:", type(error).__name__, "-", error)

try:
    print(int("not a number"))
except ValueError as error:
    print("caught:", type(error).__name__, "-", error)

try:
    print("text" + 42)
except TypeError as error:
    print("caught:", type(error).__name__, "-", error)

try:
    print(undefined_variable)
except NameError as error:
    print("caught:", type(error).__name__, "-", error)

caught: ZeroDivisionError - division by zero
caught: IndexError - list index out of range
caught: KeyError - 'b'
caught: ValueError - invalid literal for int() with base 10: 'not a number'
caught: TypeError - can only concatenate str (not "int") to str
caught: NameError - name 'undefined_variable' is not defined


## try / except / else / finally
* the basic building block of error handling is the `try` statement
* `try:` contains the code that might fail
* `except SomeError:` catches a specific exception type and runs instead of crashing
* `else:` (optional) runs only if the try block did NOT raise any exception
* `finally:` (optional) always runs, no matter if an exception happened or not - typically used for cleanup (closing a file, a network connection, ...)

In [2]:
def safe_divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print("cant divide by zero!")
        return None
    else:
        print("division worked without any problem")
        return result
    finally:
        print("safe_divide finished (this always runs)")

print(safe_divide(10, 2))
print(safe_divide(10, 0))

division worked without any problem
safe_divide finished (this always runs)
5.0
cant divide by zero!
safe_divide finished (this always runs)
None


In [3]:
# it is possible (and recommended) to catch multiple, specific exception types separately
def parse_and_divide(a_text, b_text):
    try:
        a = int(a_text)
        b = int(b_text)
        return a / b
    except ValueError:
        print("one of the inputs was not a valid integer")
    except ZeroDivisionError:
        print("the second number cant be zero")

parse_and_divide("10", "abc")
parse_and_divide("10", "0")
parse_and_divide("10", "5")

# it is also possible to catch multiple exception types with the same handler, grouped in a tuple
def risky_lookup(data, key, index):
    try:
        return data[key][index]
    except (KeyError, IndexError) as error:
        print("lookup failed:", type(error).__name__, error)
        return None

risky_lookup({"numbers": [1, 2, 3]}, "numbers", 10)   # IndexError
risky_lookup({"numbers": [1, 2, 3]}, "letters", 0)    # KeyError

# a bare "except:" without any type catches EVERYTHING, including exceptions you didnt expect and dont know how to handle
# this is considered bad practice - always be as specific as possible about which exception you expect

one of the inputs was not a valid integer
the second number cant be zero
lookup failed: IndexError list index out of range
lookup failed: KeyError 'letters'


## raising your own exceptions
* with `raise` we can trigger an exception ourselves, for example if our own function receives invalid input
* python has many builtin exception types (`ValueError`, `TypeError`, `KeyError`, ...) that we can reuse for our own errors instead of inventing something new every time

In [4]:
def set_age(age):
    if not isinstance(age, int):
        raise TypeError(f"age must be an int, got {type(age).__name__}")
    if age < 0:
        raise ValueError("age cant be negative")
    return age

try:
    set_age(-5)
except ValueError as error:
    print("caught our own raised error:", error)

try:
    set_age("old")
except TypeError as error:
    print("caught our own raised error:", error)

print(set_age(30))

caught our own raised error: age cant be negative
caught our own raised error: age must be an int, got str
30


## custom exception classes
* for bigger applications it is common to define your own exception types by inheriting from the builtin `Exception` class (we will fully cover classes and inheritance in the next lesson, this is a small preview)
* custom exceptions make it possible for code calling your functions to catch exactly YOUR error types, separately from generic builtin errors

In [5]:
class InsufficientFundsError(Exception):
    """Raised when a withdrawal is bigger than the available balance."""
    pass

def withdraw(balance, amount):
    if amount > balance:
        raise InsufficientFundsError(f"cant withdraw {amount}, balance is only {balance}")
    return balance - amount

try:
    withdraw(100, 150)
except InsufficientFundsError as error:
    print("caught custom exception:", error)

# because InsufficientFundsError inherits from Exception, it can also still be caught generically
try:
    withdraw(100, 150)
except Exception as error:
    print("also caught generically as Exception:", type(error).__name__, error)

caught custom exception: cant withdraw 150, balance is only 100
also caught generically as Exception: InsufficientFundsError cant withdraw 150, balance is only 100


## the exception hierarchy
* every builtin exception inherits from `BaseException`, and almost every one you will ever catch inherits from `Exception`
* this means: `except Exception:` catches basically every "normal" error, but NOT things like `KeyboardInterrupt` (ctrl+c) or `SystemExit` - which is intentional, you usually dont want to accidentally swallow those
* order matters when catching multiple related exceptions: put more specific exception types BEFORE more general ones, python checks except-blocks top to bottom and stops at the first match

In [6]:
# ValueError is more specific than Exception, so it must come first
def demo_order(value):
    try:
        if value < 0:
            raise ValueError("negative value")
        return 100 / value
    except ValueError as error:
        print("specific ValueError handler:", error)
    except Exception as error:
        print("generic Exception handler:", type(error).__name__, error)

demo_order(-1)  # caught by the specific ValueError handler
demo_order(0)   # ZeroDivisionError, falls through to the generic Exception handler

specific ValueError handler: negative value
generic Exception handler: ZeroDivisionError division by zero


## context managers and finally for cleanup
* we already used `with open(...) as file:` in the previous lesson - this IS a context manager and it guarantees the file gets closed, even if an exception happens inside the block
* this makes context managers the preferred cleanup tool over a manual try/finally whenever one is available

In [7]:
# without a context manager we would need to manually guarantee cleanup with try/finally
file = open("temp_demo.txt", "w")
try:
    file.write("some content")
    # imagine an error happens here
finally:
    file.close()  # this always runs, so the file is never left open by accident

import os
os.remove("temp_demo.txt")

# the "with" statement does exactly the same thing, just shorter and safer (less room to forget the finally block)
with open("temp_demo.txt", "w") as file:
    file.write("some content")
os.remove("temp_demo.txt")

print("both approaches close the file safely, 'with' is just less error-prone")

both approaches close the file safely, 'with' is just less error-prone


## assert - a debugging helper, not real error handling
* `assert condition, "message"` raises an `AssertionError` if the condition is False
* asserts are meant for catching programmer mistakes during development/testing, NOT for validating real user input (asserts can be globally disabled with the `-O` "optimize" flag when running python, so never rely on them for security or important business logic checks)

In [8]:
def average(numbers):
    assert len(numbers) > 0, "cant calculate average of an empty list"
    return sum(numbers) / len(numbers)

print(average([1, 2, 3, 4]))

try:
    average([])
except AssertionError as error:
    print("caught assertion:", error)

2.5
caught assertion: cant calculate average of an empty list


## quick comparison to other languages
* python's `try/except/else/finally` is closest to JavaScript's `try/catch/finally` (JS has no `else` block) and Java/C#'s `try/catch/finally`
* python catches exceptions by TYPE (`except ValueError:`), same idea as Java/C#/C++ `catch (SomeExceptionType e)`; JavaScript by contrast has only one generic `catch (error)` block and you check the error type yourself inside it
* rust and Go deliberately do NOT use exceptions at all for expected errors - Go functions return an explicit `(result, error)` pair you must check yourself, Rust functions return a `Result<T, E>` enum that forces you to handle both the success and failure case at compile time; python's exception-based style is more forgiving to write but easier to accidentally forget a case
* python's `raise SomeError("message")` corresponds to JavaScript's `throw new Error("message")`, Java/C#'s `throw new SomeException("message")`
* python's `assert` is similar to Java's `assert` keyword and C/C++'s `assert()` macro - all three share the same "development-time only, can be disabled" nature

## Exercises
1. write a function `safe_int(text)` that tries to convert `text` to an `int` and returns it; if conversion fails, catch the exception and return `None` instead
2. write a custom exception class `NegativeValueError` and a function `square_root(number)` that raises it if `number` is negative, otherwise returns `number ** 0.5`
3. write a function `get_item(my_list, index)` that returns `my_list[index]`, but catches an `IndexError` and returns the string `"index out of range"` instead of crashing
4. write a function `divide_all(numbers, divisor)` that divides every number in the list by divisor and returns the results as a new list; use try/except/finally so that "done dividing" is always printed, even if a `ZeroDivisionError` happens

In [9]:
# TODO: solve exercise 1 here (safe_int)


# TODO: solve exercise 2 here (NegativeValueError + square_root)


# TODO: solve exercise 3 here (get_item)


# TODO: solve exercise 4 here (divide_all with try/except/finally)

### sample solutions
* try the exercises above yourself first, only look here if you are stuck or want to compare your approach

In [10]:
# sample solution 1
def safe_int(text):
    try:
        return int(text)
    except ValueError:
        return None

print(safe_int("42"), safe_int("not a number"))

# sample solution 2
class NegativeValueError(Exception):
    pass

def square_root(number):
    if number < 0:
        raise NegativeValueError(f"cant take the square root of a negative number: {number}")
    return number ** 0.5

print(square_root(16))
try:
    square_root(-4)
except NegativeValueError as error:
    print("caught:", error)

# sample solution 3
def get_item(my_list, index):
    try:
        return my_list[index]
    except IndexError:
        return "index out of range"

print(get_item([1, 2, 3], 1))
print(get_item([1, 2, 3], 99))

# sample solution 4
def divide_all(numbers, divisor):
    try:
        return [number / divisor for number in numbers]
    except ZeroDivisionError:
        print("divisor cant be zero")
        return []
    finally:
        print("done dividing")

print(divide_all([10, 20, 30], 2))
print(divide_all([10, 20, 30], 0))

42 None
4.0
caught: cant take the square root of a negative number: -4
2
index out of range
done dividing
[5.0, 10.0, 15.0]
divisor cant be zero
done dividing
[]


* congratulation you finished this lesson

## what we learned
* what exceptions are and why unhandled exceptions crash a program
* try/except/else/finally and catching specific vs. multiple exception types
* raising builtin exceptions and defining your own custom exception classes
* the exception hierarchy and why order matters when catching multiple types
* context managers as the safe alternative to manual try/finally cleanup
* assert as a development-time debugging helper, not a replacement for real validation